In [ ]:
%load_ext autoreload 
#hopefully this will reload the modules when they are changed specifically when i change the plotting modules
%autoreload 2

In [ ]:
from myproject import GalaxyGroup, Subhalo, ListGalaxyGroup, AstroPlotter
import h5py as h5
import numpy as np

In [ ]:
sim = 'TNG300-1'
scratchDataDirc = f'/scratch/poulin.al/lopsided/{sim}/data'
scratchPlotDirc = f'/scratch/poulin.al/lopsided/{sim}/plots'
localDataDirc = f'/Users/alexpoulin/Library/CloudStorage/OneDrive-NortheasternUniversity/TGB–Data'


## load data from hdf5 file


In [ ]:
data_file = scratchDataDirc + f'/galaxy_data_{sim}.hdf5'
# data_file = localDataDirc + f'/galaxy_data_{sim}.hdf5'
with h5.File(data_file, 'r') as f:
    list_of_galaxy_groups = ListGalaxyGroup.from_hdf5(f)
print(f'Loaded galaxy data from {data_file}')

In [ ]:
#filter to groups with stellar mass > 10^{14} Msun
filtered_galaxy_groups = list_of_galaxy_groups.filterSubhalos(minGGMass=1e14)
# print(f"len filtered: {len(filtered_galaxy_groups)}")
filtered_list_of_galaxy_groups = ListGalaxyGroup(headerInformation = list_of_galaxy_groups.getHeaderInformation, listGalaxyGroups=filtered_galaxy_groups)

In [ ]:
#check the number of galaxy groups loaded
print(f'Number of galaxy groups loaded: {filtered_list_of_galaxy_groups.getNumGalaxyGroups()}')
print(f"Number of subhalos in GG 0: {filtered_list_of_galaxy_groups.getGalaxyGroupI(0).getNumSubhalos()}")
print(f"Number of subhalos in GG 10: {filtered_list_of_galaxy_groups.getGalaxyGroupI(10).getNumSubhalos()}")
print(f' Range of satellites: {filtered_list_of_galaxy_groups.getRangeOfNumSubhalos()}')

## Make plots for Pairwise Polar Differences

In [ ]:
filtered_list_of_galaxy_groups.compute_all_pairwise_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar')

In [ ]:
# polar_bin_centers, pairwise_polar_differences = list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(parallelize=False, tempSaveDir=localDataDirc)
polar_bin_centers, pairwise_polar_differences = filtered_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar')
print('Computed pairwise polar differences between satellite galaxies.')
print(polar_bin_centers)

In [ ]:
prob_polar_plotter = AstroPlotter()
prob_polar_plotter.scatter_plot(
    polar_bin_centers, 
    pairwise_polar_differences,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar_difference_{sim}.png',
    grid=True,
)

In [ ]:
# save bin centers and probabilities to text file
output_data_file = scratchDataDirc + f'/pairwise_polar_difference_{sim}.txt'
print(f"data to be saved: {np.column_stack((polar_bin_centers, pairwise_polar_differences))}")
np.savetxt(output_data_file, np.column_stack((polar_bin_centers, pairwise_polar_differences)), header='Pairwise Polar Difference (degrees)    Probability Density')

## Make plots for Mean Resultant Length (MRL) directionality

In [ ]:
filtered_list_of_galaxy_groups.compute_all_MRL_directionality(parallelize=False, tempSaveDir=f'{scratchDataDirc}/MRL_directionality')

In [ ]:
MRL_bin_centers, MRL_directionality = filtered_list_of_galaxy_groups.compute_probablity_distribution_of_MRL_directionality(parallelize=False, tempSaveDir=f'{scratchDataDirc}/MRL_directionality')
print('Computed MRL directionality for galaxy groups.')


In [ ]:
prob_MRL_plotter = AstroPlotter()
prob_MRL_plotter.scatter_plot(
    MRL_bin_centers, 
    MRL_directionality,
    xlabel='MRL Directionality',
    ylabel='Probability Density',
    title=f'Probability MRL Directionality Distribution for {sim}',
    # ylim=(0,4)
    output_filename=scratchPlotDirc + f'/MRL_directionality_{sim}.png',
    grid=True
)

In [ ]:
#save bin centers and probabilities to text file
output_data_file_MRL = scratchDataDirc + f'/MRL_directionality_{sim}.txt'
print(f"data to be saved: {np.column_stack((MRL_bin_centers, MRL_directionality))}")
np.savetxt(output_data_file_MRL, np.column_stack((MRL_bin_centers, MRL_directionality)), header='MRL Directionality    Probability Density')

## make plots for red vs blue galaxies

In [ ]:
filteredRedGalaxies = filtered_list_of_galaxy_groups.filterSubhalos(redGalaxies=True)
print(f'Number of galaxy groups with only red satellites: {filteredRedGalaxies.getNumGalaxyGroups()}')
filteredBlueGalaxies = filtered_list_of_galaxy_groups.filterSubhalos(blueGalaxies=True)
print(f'Number of galaxy groups with only blue satellites: {filteredBlueGalaxies.getNumGalaxyGroups()}')

filtered_red_list_of_galaxy_groups = ListGalaxyGroup(headerInformation = filtered_list_of_galaxy_groups.getHeaderInformation, listGalaxyGroups=filteredRedGalaxies)
filtered_blue_list_of_galaxy_groups = ListGalaxyGroup(headerInformation = filtered_list_of_galaxy_groups.getHeaderInformation, listGalaxyGroups=filteredBlueGalaxies)
filtered_red_list_of_galaxy_groups.compute_all_pairwise_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_red')
filtered_blue_list_of_galaxy_groups.compute_all_pairwise_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_blue')


In [ ]:
polar_bin_centers_red, pairwise_polar_differences_red = filtered_red_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_red')
polar_bin_centers_blue, pairwise_polar_differences_blue = filtered_blue_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_blue')

In [ ]:
print('Computed pairwise polar differences between red and blue satellite galaxies.')
prob_polar_red_blue_plotter = AstroPlotter()
prob_polar_red_blue_plotter.scatter_plot(
    polar_bin_centers_red, 
    pairwise_polar_differences_red,
    ax=None,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for Red and Blue Satellites in {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar_difference_red_blue_{sim}.png',
    grid=True,
)
prob_polar_red_blue_plotter.scatter_plot(
    polar_bin_centers_blue, 
    pairwise_polar_differences_blue,
    ax=None,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for Red and Blue Satellites in {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar_difference_red_blue_{sim}.png',
    grid=True,
)

## make plots for 150 vs 50 member galaxies

In [ ]:
filteredGT150 = filtered_list_of_galaxy_groups.filterGalaxyGroupsMinNumSubhalos(150)
filtered_GT150_list_of_galaxy_groups = ListGalaxyGroup(headerInformation = filtered_list_of_galaxy_groups.getHeaderInformation, listGalaxyGroups=filteredGT150)
print(f'Number of galaxy groups with more than 150 satellites: {filtered_GT150_list_of_galaxy_groups.getNumGalaxyGroups()}')
filteredLT50 = filtered_list_of_galaxy_groups.filterGalaxyGroupsMaxNumSubhalos(50)
filtered_LT50_list_of_galaxy_groups = ListGalaxyGroup(headerInformation = filtered_list_of_galaxy_groups.getHeaderInformation, listGalaxyGroups=filteredLT50)
print(f'Number of galaxy groups with less than 50 satellites: {filtered_LT50_list_of_galaxy_groups.getNumGalaxyGroups()}')

filtered_GT150_list_of_galaxy_groups.compute_all_pairwise_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_GT150')
filtered_LT50_list_of_galaxy_groups.compute_all_pairwise_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_LT50')

In [ ]:
print('Computed pairwise polar differences for galaxy groups with >150 and <50 satellites.')
polar_bin_centers_GT150, pairwise_polar_differences_GT150 = filtered_GT150_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_GT150')
polar_bin_centers_LT50, pairwise_polar_differences_LT50 = filtered_LT50_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_LT50')

In [ ]:
prob_polar_GT150_LT50_plotter = AstroPlotter()
prob_polar_GT150_LT50_plotter.scatter_plot(
    polar_bin_centers_GT150, 
    pairwise_polar_differences_GT150,
    ax=None,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for >150 and <50 Satellites in {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar_difference_GT150_LT50_{sim}.png',
    grid=True,
)
prob_polar_GT150_LT50_plotter.scatter_plot(
    polar_bin_centers_LT50, 
    pairwise_polar_differences_LT50,
    ax=None,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for >150 and <50 Satellites in {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar_difference_LT50_{sim}.png',
    grid=True,
)

## make plots for <35% vs >65% R200 plots

In [ ]:
filteredLT35R200 = filtered_list_of_galaxy_groups.filterSubhalos(withinXPercentR200=[0,35])
filtered_LT35R200_list_of_galaxy_groups = ListGalaxyGroup(headerInformation = filtered_list_of_galaxy_groups.getHeaderInformation, listGalaxyGroups=filteredLT35R200)
print(f'Number of galaxy groups with satellites within 35% R200: {filtered_LT35R200_list_of_galaxy_groups.getNumGalaxyGroups()}')
filtered_GT65R200 = filtered_list_of_galaxy_groups.filterSubhalos(withinXPercentR200=[65,100])
filtered_GT65R200_list_of_galaxy_groups = ListGalaxyGroup(headerInformation = filtered_list_of_galaxy_groups.getHeaderInformation, listGalaxyGroups=filtered_GT65R200)
print(f'Number of galaxy groups with satellites within 65-100% R200: {filtered_GT65R200_list_of_galaxy_groups.getNumGalaxyGroups()}')

filtered_LT35R200_list_of_galaxy_groups.compute_all_pairwise_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_LT35R200')
filtered_GT65R200_list_of_galaxy_groups.compute_all_pairwise_polar_differences(parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_GT65R200')

In [ ]:
polar_bin_centers_LT35R200, pairwise_polar_differences_LT35R200 = filtered_LT35R200_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_LT35R200')
polar_bin_centers_GT65R200, pairwise_polar_differences_GT65R200 = filtered_GT65R200_list_of_galaxy_groups.compute_probablity_distribution_of_polar_differences(bins = np.arange(0, 185, 10), parallelize=False, tempSaveDir=f'{scratchDataDirc}/pairwise_polar_GT65R200')

In [ ]:
prob_polar_GT150_LT50_plotter = AstroPlotter()
prob_polar_GT150_LT50_plotter.scatter_plot(
    polar_bin_centers_GT150, 
    pairwise_polar_differences_GT150,
    ax=None,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for >150 and <50 Satellites in {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar_difference_GT150_LT50_{sim}.png',
    grid=True,
)
prob_polar_GT150_LT50_plotter.scatter_plot(
    polar_bin_centers_LT50, 
    pairwise_polar_differences_LT50,
    ax=None,
    xlabel='Pairwise Polar Difference (degrees)',
    ylabel='Probability Density',
    title=f'Probability Pairwise Polar Difference Distribution for >150 and <50 Satellites in {sim}',
    # ylim = (0, 0.01),
    output_filename=scratchPlotDirc + f'/pairwise_polar_difference_LT50_{sim}.png',
    grid=True,
)